# Vista Semántica: Evolución Temporal de Vistas (`v_evolucion_temporal_vistas`)

Esta vista está optimizada para el **gráfico de líneas** del dashboard:
- **Eje X**: Tiempo (`fecha`, `anio`, `mes`, `semana_anio`).
- **Eje Y**: Vistas (`total_vistas`, `promedio_vistas`).
- **Métricas complementarias**: `total_likes`, `cantidad_videos_publicados`, `engagement_promedio`.

Incluye ejemplos de parametrización mediante filtros de fecha y widgets de Databricks para cambiar variables interactivamente.

**Schema:** `tiktok_data_eng.semantica`

In [0]:
%sql
-- Creación de la vista en schema semantica
CREATE OR REPLACE VIEW tiktok_data_eng.semantica.v_evolucion_temporal_vistas AS
SELECT 
    fe.fecha,
    fe.anio,
    fe.mes,
    fe.nombre_mes,
    fe.semana_anio,
    fe.dia_semana,
    fe.nombre_dia,
    fe.es_fin_de_semana,
    SUM(f.plays) AS total_vistas,
    ROUND(AVG(f.plays), 2) AS promedio_vistas,
    SUM(f.likes) AS total_likes,
    COUNT(f.video_id) AS cantidad_videos_publicados,
    ROUND(AVG(f.engagement_rate), 4) AS engagement_promedio
FROM tiktok_data_eng.gold.fct_video_metricas f
JOIN tiktok_data_eng.gold.dim_fecha fe ON f.fecha_id = fe.fecha_id
GROUP BY 
    fe.fecha, 
    fe.anio, 
    fe.mes, 
    fe.nombre_mes, 
    fe.semana_anio, 
    fe.dia_semana, 
    fe.nombre_dia, 
    fe.es_fin_de_semana
ORDER BY fe.fecha ASC;

### Consulta Parametrizada por Rango de Fechas (Diaria)

In [0]:
%sql
-- Consulta parametrizable para el gráfico de líneas (Eje X: fecha, Eje Y: total_vistas)
SELECT 
    fecha,
    total_vistas,
    promedio_vistas,
    cantidad_videos_publicados
FROM tiktok_data_eng.semantica.v_evolucion_temporal_vistas
WHERE fecha >= '2024-01-01'
ORDER BY fecha ASC;

### Consulta Parametrizada por Agregación Mensual (Eje X = Año-Mes, Eje Y = Vistas)

In [0]:
%sql
-- Vista agregada por mes para suavizar la tendencia temporal en el gráfico
SELECT 
    CONCAT(anio, '-', LPAD(mes, 2, '0')) AS periodo_mes,
    SUM(total_vistas) AS total_vistas_mes,
    ROUND(AVG(promedio_vistas), 2) AS promedio_vistas_mes,
    SUM(cantidad_videos_publicados) AS total_videos_mes
FROM tiktok_data_eng.semantica.v_evolucion_temporal_vistas
GROUP BY anio, mes
ORDER BY anio ASC, mes ASC;